# job6_A1280 — runtime: **T4 GPU**

Runs: A1280_f0, A1280_f1


# PPE revision runner — one notebook per job

Paper revision (ICCEAI 2026, paper 8): every model is trained with the same
recipe and scored with one harness. This notebook runs **one job**; the job
name is set in the next cell (the generated copies in `training/notebooks/`
have it preset).

| JOB | Runtime | What it does |
|---|---|---|
| `job0` | CPU | builds `prepared/pool.zip` (run first, once) + re-scores the old model |
| `job1` | T4 GPU | B seeds 0-2, A folds 0-2, latency |
| `job2` | T4 GPU | S seeds 0-2 |
| `job3` | T4 GPU | A folds 3-4 |
| `job4` | T4 GPU | P seeds 0-2 |
| `collect` | CPU | all tables + figures into `results/revision/final/` |

**Before job0:** upload `OCP.yolov8.zip` (the 207 un-augmented OCP frames) to
`MyDrive/ppe-training/raw_uploads/`.

**Then: Runtime → Run all.** Disconnected? Reconnect and Run all again —
finished runs are skipped, an unfinished run resumes from its last epoch.


In [ ]:
JOB = 'job6'  # preset by scripts/make_job_notebooks.py


## Setup — mount Google Drive

Datasets, checkpoints and results live on Drive under `ppe-training/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Setup — clone code from GitHub + install dependencies

Needs a Colab Secret `GITHUB_TOKEN` with read access to the repository.


In [ ]:
# Clone the private pipeline repo from GitHub; keep durable artifacts on Drive.
from google.colab import userdata
from pathlib import Path
import importlib.util
import os
import shutil
import subprocess
import tempfile

REPO_URL = 'https://github.com/YasserDalali/ppe-cv.git'
REPO_BRANCH = 'revision'
REPO_DIR = Path('/content/ppe-cv')
CODE = Path('/content/ppe-cv/training')
DRIVE_ROOT = Path('/content/drive/MyDrive/ppe-training')
REQS = CODE / 'requirements.txt'

try:
    github_token = userdata.get('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError(
        'Add a Colab secret named GITHUB_TOKEN with read access to YasserDalali/ppe-cv.'
    ) from exc
github_token = (github_token or '').strip()
if not github_token:
    raise RuntimeError('Colab secret GITHUB_TOKEN is empty.')

# Prefer GIT_ASKPASS over Bearer headers / token URLs: more reliable for
# fine-grained PATs and keeps the token out of argv / clone logs.
askpass = Path(tempfile.mkdtemp(prefix='git-askpass-')) / 'askpass.sh'
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    '  *Username*) printf %s "x-access-token" ;;\n'
    '  *) printf %s "$GIT_PASSWORD" ;;\n'
    'esac\n'
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({
    'GIT_ASKPASS': str(askpass),
    'GIT_TERMINAL_PROMPT': '0',
    'GIT_PASSWORD': github_token,
})
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
shutil.rmtree(REPO_DIR, ignore_errors=True)
clone = subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)],
    env=git_env, capture_output=True, text=True,
)
askpass.unlink(missing_ok=True)
askpass.parent.rmdir()
del github_token, git_env
if clone.returncode != 0:
    err = (clone.stderr or clone.stdout or '').strip()
    raise RuntimeError(
        'git clone failed. Check: (1) Colab secret GITHUB_TOKEN has Notebook access, '
        '(2) fine-grained token includes repo YasserDalali/ppe-cv with Contents: Read-only, '
        f'(3) branch {REPO_BRANCH} exists.\nGit said:\n{err}'
    )

if not REQS.is_file():
    raise FileNotFoundError(f'GitHub clone is missing {REQS}')
subprocess.run(['pip', 'install', '-q', '-r', str(REQS)], check=True)
if importlib.util.find_spec('ultralytics') is None:
    raise RuntimeError(f'Failed to install deps from {REQS}')
print(f'code ready from {REPO_URL}@{REPO_BRANCH}')
print('deps ready')


## Configuration

Defaults match the paper recipe (YOLOv8n, 640, batch 16, 50 epochs). Secrets
(`secrets.env` on Drive) are only needed by job0, to fetch any public source
not already cached in `raw/`.


In [ ]:
import sys
sys.path.insert(0, str(CODE / 'src'))
from pathlib import Path
from ppe.config import JOBS, Config, SecretsError, load_secrets

cfg = Config(drive_root=DRIVE_ROOT, work_root=Path('/content/ppe-work'))
try:
    secrets = load_secrets(cfg)
except SecretsError:
    secrets = None
    print('no secrets.env — fine unless the pool still has to be built (job0)')
if JOB not in JOBS:
    raise ValueError(f'JOB must be one of {sorted(JOBS)}')
if JOB.startswith('job') and JOB != 'job0':
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('This job trains models: Runtime -> Change runtime type -> GPU')
print('JOB', JOB, '->', JOBS[JOB] or '(no training)')


## Run the job

Everything is written to Drive: `runs/<run_id>/` (checkpoints) and
`results/revision/<run_id>/` (predictions + scores).


In [ ]:
from ppe.jobs import run_job
run_job(cfg, JOB, secrets)
